## LangChain

Required env vars:
- `OPENAI_API_KEY`
- `GOOGLE_API_KEY` (for Google Gemini integration cells)
- `PPLX_API_KEY` (for Perplexity integration cells)

### Basic `ChatOpenAI` invocation

Minimal `ChatOpenAI` call showing common constructor parameters (several left commented out for reference) and a system/human message pair.

Reference: https://docs.langchain.com/oss/python/integrations/chat/openai

In [ ]:
from langchain_openai.chat_models import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-nano",
    # stream_usage=True,
    temperature=0.7,
    # max_tokens=None,
    timeout=None,
    reasoning_effort="low",
    max_retries=1,
    # api_key="...",  # If you prefer to pass api key in directly
    # base_url="...",
    # organization="...",
    # other params...
)

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

print(ai_msg.text)

### Responses API + tool binding (web search)

Uses `ChatOpenAI(..., use_responses_api=True)` to switch from the Chat Completions API to OpenAI's newer Responses API, enabling agentic features like web search via `bind_tools`. Note: the `sonar` model doesn't support the Responses API.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", use_responses_api=True)

tool = {"type": "web_search_preview"}
llm_with_tools = llm.bind_tools([tool])

response = llm_with_tools.invoke("What was a single positive news story from today?")
print(response.text)

response.content_blocks

### Prompt templates (`langchain-core`)

Builds a `ChatPromptTemplate` with system/human/ai turns and variable placeholders, then invokes it twice with different inputs.

Reference: https://reference.langchain.com/python/langchain-core

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

template = ChatPromptTemplate(
    [
        ("system", "You are a helpful AI bot. Your name is {name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "{user_input}"),
    ]
)

prompt_value = template.invoke(
    {
        "name": "Bob",
        "user_input": "What is your name?",
    }
)

messages = template.invoke({"name": "Alice", "user_input": "How do you work?"})
print(messages)

### `ChatPerplexity` — basic invocation

References:
- https://docs.langchain.com/oss/python/integrations/chat/perplexity
- https://reference.langchain.com/python/langchain-perplexity

In [ ]:
import os
from dotenv import load_dotenv
from langchain_perplexity import ChatPerplexity

load_dotenv()
api_key = os.getenv("PPLX_API_KEY")

if not api_key:
    raise ValueError("PPLX_API_KEY environment variable not set")
model = ChatPerplexity(model="sonar-pro", temperature=0.7, max_retries=1)
response = model.invoke("What is the capital of France?")
print(response.text)
print("\n\n")
print(response.response_metadata)

### `ChatPerplexity` — `use_responses_api` is not supported

Unlike `ChatOpenAI`, `ChatPerplexity` doesn't support the `use_responses_api` parameter — passing it raises an error, so it's left commented out below as a reference.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_perplexity import ChatPerplexity

load_dotenv()
api_key = os.getenv("PPLX_API_KEY")
if not api_key:
    raise ValueError("PPLX_API_KEY environment variable not set")

model = ChatPerplexity(model="sonar-pro")  # use_responses_api=True is invalid here
response = model.invoke("What is the capital of France?")
print(response.text)
print("\n\n")
print(response.response_metadata)

### Chain composition with the pipe operator

Demonstrates provider-agnostic model swapping: a prompt template piped into `ChatOpenAI` via `prompt | llm`, so LangChain standardizes the interface and swapping providers only requires changing the model object.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(temperature=0.7, model="gpt-4o")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that translates English to French."),
        ("human", "{input}"),
    ]
)

chain = prompt | llm

response = chain.invoke({"input": "I love programming in Python."})

print(response.content)

### Structured output extraction (`with_structured_output`)

Uses `with_structured_output` to reliably extract structured data from unstructured text — LangChain uses OpenAI's function/tool calling under the hood to guarantee schema compliance.

Reference: https://python.langchain.com/docs/concepts/structured_outputs/

*(Originally written with GitHub Copilot, as part of this repo's AI-assisted development exploration.)*

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List
from langchain_openai import ChatOpenAI


class PersonInfo(BaseModel):
    name: str = Field(description="The full name of the person")
    age: Optional[int] = Field(
        default=None, description="The age of the person, if mentioned"
    )
    occupation: Optional[str] = Field(
        default=None, description="The person's job or occupation"
    )
    skills: List[str] = Field(
        description="List of skills or technologies the person knows"
    )


llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Bind the schema — the LLM will always return a valid PersonInfo object
structured_llm = llm.with_structured_output(PersonInfo)

text = """
    Sarah is a 32-year-old software engineer who specialises in backend development.
    She has strong experience with Python, FastAPI, PostgreSQL, and Docker, and has
    recently been learning Rust and Kubernetes.
"""

result = structured_llm.invoke(f"Extract the person details from this text:\n{text}")

print(f"Name:       {result.name}")
print(f"Age:        {result.age}")
print(f"Occupation: {result.occupation}")
print(f"Skills:     {', '.join(result.skills)}")